Notebook de Exploração inicial no dataset:
Análise de tipos de colunas, formatos, compatibilidade dos dados... etc.    


In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

# Pergunta alvo do projeto: O tempo de entrega influencia na avaliação dos clientes?

# Dossiê tabela Orders
# Pergunta alvo do projeto: O tempo de entrega influencia na avaliação dos clientes?
99441 linhas
8 colunas
Dtypes - todas as colunas vieram como object

colunas para análise e conversão:
- order_id - mantém object
- order_status - mantém object
- order_delivered_customer_date > datetime
- order_purchase_timestamp > datetime

entregues (status = delivered) - 96478
pedidos com datas de entrega - 96476

8 pedidos estão com status delivered mas não possuem data de entrega
6 pedidos com status cancelado possuem data de entrega


-----------------------------------
procedimento:

o tempo de entrega é data_de_entrega − data_da_compra

filtro inicial: somente entregas com status = delivered, pois tempo de 
entrega só existe pra pedido entregue! 

8 pedidos sem data de entrega serão propositalmente desconsiderados 
- não há como medir o tempo de entrega!


Escopo v1: tempo de entrega × nota. Extensão mapeada (fora do v1): 
atraso vs. prazo estimado (entrega_real − entrega_estimada) — hipótese 
de que a quebra da promessa pesa mais que o tempo absoluto. Fica como 
róximo incremento

# Dossiê da tabela reviews

99.224 linhas 
7 colunas

Dtypes — somente a coluna review_score veio com o tipo certo - int64, todas as outras: object

colunas para análise e conversão:

- review_id (em busca de ids repetidos) - mantém object
- order_id (chave) - mantém object
- review_score - mantém int
- review_answer_timestamp > datetime

PS: as colunas que apresentam nulos são as colunas de comentários e 
títulos dos comentários dos reviews, não sendo as nossas colunas alvos 
para a análise principal do projeto.

As colunas principais: order_id e review_score  estão completas

Há 271 pedidos a mais que reviews no total; a causa(pedido sem review? 
review duplicado? órfão?) só o cruzamento dos order_id confirma.

551 reviews com ordem de pedido duplicados (com keep=False, ~1.100+ linhas envolvidas) 
- review_id distintos para o mesmo pedido: não são linhas duplicadas!
- múltiplas avaliações do mesmo pedido, as vezes concordando e as vezes discordando entre si
- Causa provável: pedidos multi-item / re-avaliações
- drop_duplicates() seco NÃO pega 
- "O que conta como a avaliação de um pedido?" 
# o dono dessa definição é a área de negócio, não o engenheiro 
- nesses casos, vamos utilizar a coluna review_answer_timestamp: priorizar o sinal de fidelidade 
do cliente (muito provavelmente o último review dele é o que sinaliza se ele tornará a comprar 
conosco) para definir qual review_score será considerado.
Trade consciente: a última review está mais distante da entrega no tempo, então embute ruído 
pós-entrega (suporte, troca). Aceito e documentado.

In [2]:
# Dociê tabela Orders
print(orders.shape)
print(orders.info())
print(orders.isna().sum())
orders['order_status'].value_counts()


(99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB
None
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
orde

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [3]:
# pedidos com status delivered sem data de entrega
orders_entregue = orders[orders['order_status'] == 'delivered'].copy()
fantasmas = orders_entregue[orders_entregue['order_delivered_customer_date'].isnull()]
fantasmas.head(10)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [4]:
# pedidos não delivered mas com data de entrega
orders_no_delivered = orders[orders['order_status'] != 'delivered']
fantasmas2 = orders_no_delivered[orders_no_delivered['order_delivered_customer_date'].notna()]
print(fantasmas2.info())
fantasmas2.head(100)


<class 'pandas.DataFrame'>
Index: 6 entries, 2921 to 94399
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       6 non-null      str  
 1   customer_id                    6 non-null      str  
 2   order_status                   6 non-null      str  
 3   order_purchase_timestamp       6 non-null      str  
 4   order_approved_at              6 non-null      str  
 5   order_delivered_carrier_date   6 non-null      str  
 6   order_delivered_customer_date  6 non-null      str  
 7   order_estimated_delivery_date  6 non-null      str  
dtypes: str(8)
memory usage: 1.4 KB
None


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


In [5]:
# Dociê tabela reviews
print(reviews.shape)
print(reviews.info())
print(reviews.isna().sum())
reviews['review_score'].value_counts()

(99224, 7)
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB
None
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [6]:
# tabela de duplicados
dups = reviews[reviews['order_id'].duplicated(keep=False)].sort_values('order_id')
print(dups.shape)
dups.head(20)

(1098, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


In [7]:
import pandas as pd
import sys
print(pd.__version__)
print(sys.executable)   # tem que terminar em .../olist-pipeline/.venv/bin/python

3.0.3
/home/vlad/projetos/olist-pipeline/.venv/bin/python
